# Day 3 — Probability and Generative Modelling

**Workshop:** Mathematical Foundations of Modern AI

Companion notebook to the Day 3 lecture notes. Four parts:

1. Sampling from Gaussians, fitting a Gaussian by MLE, computing KL divergence by hand and by Monte Carlo.
2. A small variational autoencoder (VAE) on MNIST: latent-space visualization and a latent walk.
3. A denoising diffusion model on a 2D spiral dataset, with step-by-step visualization of forward noising and reverse generation.
4. A small diffusion model on MNIST, with sampled digits and the intermediate noise levels.

Runs on CPU. GPU optional for Part 4 (about 5 minutes faster).

---

## Part 1 — Gaussians, likelihood, KL

Generate samples, fit a Gaussian, and compute KL two ways.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)

# Truth: a 1-D Gaussian with mu_true=2.0, sigma_true=1.5
mu_true, sigma_true = 2.0, 1.5
samples = rng.normal(loc=mu_true, scale=sigma_true, size=2000)

# Maximum-likelihood estimates are the empirical mean and std
mu_hat = samples.mean()
sigma_hat = samples.std(ddof=0)
print(f"True:  mu={mu_true:.3f}, sigma={sigma_true:.3f}")
print(f"MLE:   mu={mu_hat:.3f}, sigma={sigma_hat:.3f}")

# Plot the fit
xs = np.linspace(samples.min(), samples.max(), 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples, bins=40, density=True, alpha=0.6, label="samples")
ax.plot(xs, (1 / (sigma_hat * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((xs - mu_hat) / sigma_hat) ** 2),
        "r-", lw=2, label=f"MLE fit  N({mu_hat:.2f}, {sigma_hat:.2f})")
ax.legend(); ax.set_title("MLE fit of a Gaussian to 2000 samples"); plt.tight_layout(); plt.show()

In [ ]:
# KL between two 1-D Gaussians has a closed form:
#   KL(N(mu1, s1^2) || N(mu2, s2^2)) = log(s2/s1) + (s1^2 + (mu1-mu2)^2) / (2 s2^2) - 1/2
def kl_gaussian_closed_form(mu1, s1, mu2, s2):
    return np.log(s2 / s1) + (s1**2 + (mu1 - mu2)**2) / (2 * s2**2) - 0.5

# Monte Carlo estimate: KL(q || p) = E_{x ~ q} [log q(x) - log p(x)]
def log_normal_pdf(x, mu, s):
    return -0.5 * np.log(2 * np.pi) - np.log(s) - 0.5 * ((x - mu) / s) ** 2

def kl_gaussian_monte_carlo(mu1, s1, mu2, s2, n=100_000):
    x = rng.normal(mu1, s1, size=n)
    return float(np.mean(log_normal_pdf(x, mu1, s1) - log_normal_pdf(x, mu2, s2)))

# Compare for a non-trivial pair
mu1, s1 = 0.0, 1.0
mu2, s2 = 1.0, 2.0
print(f"KL closed form:  {kl_gaussian_closed_form(mu1, s1, mu2, s2):.4f}")
print(f"KL Monte Carlo:  {kl_gaussian_monte_carlo(mu1, s1, mu2, s2):.4f}")

The two estimates agree to two or three decimal places. The closed form is exact and free; the Monte Carlo estimate has variance $O(1/\sqrt{n})$. For Gaussians we always use the closed form. For arbitrary distributions, Monte Carlo is the only option.

---

## Part 2 — A small VAE on MNIST

Encoder $q_\phi(z \mid x) = \mathcal{N}(\mu_\phi(x), \sigma_\phi^2(x))$, decoder $p_\theta(x \mid z) = \mathrm{Bern}(\sigma(\mathrm{net}(z)))$. Latent dimension 2 so we can plot it.

Loss: negative ELBO = reconstruction (binary cross-entropy) + KL to the standard Gaussian prior. Both terms have closed-form expressions for this choice of model.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

class VAE(nn.Module):
    def __init__(self, d_in: int = 784, d_hidden: int = 256, d_latent: int = 2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(d_hidden, d_latent)
        self.fc_logvar = nn.Linear(d_hidden, d_latent)
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_in)        # logits, sigmoid in the loss
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(z)
        return logits, mu, logvar

def vae_loss(x, logits, mu, logvar):
    # Reconstruction: BCE on logits, summed over pixels, averaged over batch
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.size(0)
    # KL(N(mu, exp(logvar)) || N(0, I)) closed form
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + kl, recon, kl

vae = VAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for epoch in range(5):
    vae.train()
    total = 0.0; n = 0
    for x, _ in train_loader:
        x = x.to(device)
        logits, mu, logvar = vae(x)
        loss, _, _ = vae_loss(x, logits, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * x.size(0); n += x.size(0)
    print(f"Epoch {epoch+1}: -ELBO per example = {total/n:.2f}")

In [ ]:
# Visualize the latent space, colored by digit
vae.eval()
zs, ys = [], []
with torch.no_grad():
    for x, y in test_loader:
        mu, _ = vae.encode(x.to(device))
        zs.append(mu.cpu().numpy()); ys.append(y.numpy())
zs = np.concatenate(zs); ys = np.concatenate(ys)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(zs[:, 0], zs[:, 1], c=ys, cmap="tab10", s=4, alpha=0.6)
ax.set_xlabel("z1"); ax.set_ylabel("z2"); ax.set_title("VAE latent space (encoder mean)")
plt.colorbar(sc, ax=ax, label="digit"); plt.tight_layout(); plt.show()

In [ ]:
# Latent walk: decode a path between the cluster center of digit 3 and digit 8
with torch.no_grad():
    z_a = torch.tensor(zs[ys == 3].mean(axis=0), device=device, dtype=torch.float32)
    z_b = torch.tensor(zs[ys == 8].mean(axis=0), device=device, dtype=torch.float32)
    n_steps = 10
    alphas = torch.linspace(0, 1, n_steps, device=device).unsqueeze(1)
    walk = (1 - alphas) * z_a + alphas * z_b
    decoded = torch.sigmoid(vae.decoder(walk)).cpu().numpy().reshape(n_steps, 28, 28)

fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 1.2, 1.4))
for ax, img in zip(axes, decoded):
    ax.imshow(img, cmap="gray"); ax.axis("off")
plt.suptitle("VAE latent walk from cluster of '3' to cluster of '8'"); plt.show()

In [ ]:
# Generative samples: draw z ~ N(0, I), decode
with torch.no_grad():
    z = torch.randn(16, 2, device=device)
    samples = torch.sigmoid(vae.decoder(z)).cpu().numpy().reshape(16, 28, 28)
fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img in zip(axes.flat, samples):
    ax.imshow(img, cmap="gray"); ax.axis("off")
plt.suptitle("VAE samples from the prior z ~ N(0, I)"); plt.show()

With only 2 latent dimensions, samples are blurry but recognizable as digits. Increasing the latent dimension to 16 or 32 gives much sharper samples. The cost is that you can no longer plot the latent space directly.

---

## Part 3 — A denoising diffusion model on a 2D spiral

Lowest-dimensional setting in which diffusion does something interesting. The data distribution is a 2D spiral. We train a small noise-prediction network and then sample by reversing the noising process.

In [ ]:
def make_spiral(n: int, noise: float = 0.05):
    t = np.sqrt(rng.uniform(0.25, 1.0, n)) * 3 * np.pi
    x = np.stack([t * np.cos(t), t * np.sin(t)], axis=1) / (3 * np.pi)
    return x + noise * rng.standard_normal(x.shape)

data = make_spiral(2000).astype(np.float32)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.5)
ax.set_title("Target distribution: 2D spiral"); ax.set_aspect("equal"); plt.tight_layout(); plt.show()

In [ ]:
# Beta schedule and derived quantities
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bar = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bars)

def q_sample(x0, t, eps):
    """Closed-form forward process: x_t = sqrt(alpha_bar_t) x_0 + sqrt(1-alpha_bar_t) eps."""
    return sqrt_alpha_bar[t][:, None] * x0 + sqrt_one_minus_alpha_bar[t][:, None] * eps

In [ ]:
# Visualize the forward noising chain on a few real points
data_t = torch.tensor(data[:500])
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t_show in zip(axes, [0, 25, 75, 150, 199]):
    t_idx = torch.full((data_t.size(0),), t_show, dtype=torch.long)
    eps = torch.randn_like(data_t)
    xt = q_sample(data_t, t_idx, eps).numpy()
    ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5)
    ax.set_title(f"t = {t_show}")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.suptitle("Forward noising of the spiral over time"); plt.tight_layout(); plt.show()

In [ ]:
# Sinusoidal time embedding (Vaswani et al., 2017 - same as Transformer positional encoding)
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, dtype=torch.float32) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class NoisePredictor(nn.Module):
    def __init__(self, d_data: int = 2, d_time: int = 32, d_hidden: int = 128):
        super().__init__()
        self.d_time = d_time
        self.net = nn.Sequential(
            nn.Linear(d_data + d_time, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden),         nn.SiLU(),
            nn.Linear(d_hidden, d_data)
        )

    def forward(self, x_t, t):
        emb = sinusoidal_embedding(t, self.d_time)
        return self.net(torch.cat([x_t, emb], dim=-1))

# Train
torch.manual_seed(0)
model = NoisePredictor()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
data_t = torch.tensor(data)

for step in range(3000):
    idx = torch.randint(0, data_t.size(0), (256,))
    x0 = data_t[idx]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = q_sample(x0, t, eps)
    eps_pred = model(xt, t)
    loss = F.mse_loss(eps_pred, eps)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 500 == 0:
        print(f"step {step+1}: loss = {loss.item():.4f}")

In [ ]:
# Ancestral sampling: start from noise, denoise step by step
@torch.no_grad()
def sample(model, n_samples=1000, return_chain_at=None):
    x = torch.randn(n_samples, 2)
    chain = {}
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        eps_pred = model(x, t_batch)
        # DDPM update
        coef1 = 1.0 / torch.sqrt(alphas[t])
        coef2 = betas[t] / torch.sqrt(1.0 - alpha_bars[t])
        x_mean = coef1 * (x - coef2 * eps_pred)
        if t > 0:
            noise = torch.randn_like(x)
            x = x_mean + torch.sqrt(betas[t]) * noise
        else:
            x = x_mean
        if return_chain_at is not None and t in return_chain_at:
            chain[t] = x.clone()
    return x, chain

samples, chain = sample(model, n_samples=1000, return_chain_at=[199, 150, 75, 25, 0])

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t_show in zip(axes, [199, 150, 75, 25, 0]):
    pts = chain[t_show].numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5)
    ax.set_title(f"t = {t_show}")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.suptitle("Reverse generation: noise (t=199) → spiral (t=0)"); plt.tight_layout(); plt.show()

Watch the noise organize itself into the spiral over the 200 reverse steps. The model never saw the target spiral as a label or a constraint; it only ever saw real data points with noise added, and learned to predict the noise. Sampling is the reverse trajectory.

---

## Part 4 — A small diffusion model on MNIST

Same algorithm. Larger noise predictor (flatten the image, MLP with time embedding, unflatten). Trained for a few thousand steps. Not state of the art --- but enough to generate recognizable digits.

In [ ]:
T_mnist = 200
betas_m = torch.linspace(1e-4, 0.02, T_mnist).to(device)
alphas_m = 1.0 - betas_m
alpha_bars_m = torch.cumprod(alphas_m, dim=0)

def q_sample_mnist(x0, t, eps):
    sab = torch.sqrt(alpha_bars_m[t])[:, None]
    sob = torch.sqrt(1.0 - alpha_bars_m[t])[:, None]
    return sab * x0 + sob * eps

class MNISTDiffuser(nn.Module):
    def __init__(self, d_data: int = 784, d_time: int = 64, d_hidden: int = 512):
        super().__init__()
        self.d_time = d_time
        self.net = nn.Sequential(
            nn.Linear(d_data + d_time, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden),         nn.SiLU(),
            nn.Linear(d_hidden, d_hidden),         nn.SiLU(),
            nn.Linear(d_hidden, d_data)
        )

    def forward(self, x_t, t):
        emb = sinusoidal_embedding(t, self.d_time).to(x_t.device)
        return self.net(torch.cat([x_t, emb], dim=-1))

# Re-scale MNIST to [-1, 1] for diffusion
train_loader_diff = DataLoader(train_ds, batch_size=128, shuffle=True)

diffuser = MNISTDiffuser().to(device)
opt = torch.optim.Adam(diffuser.parameters(), lr=2e-4)
diffuser.train()

n_epochs_diff = 3
for epoch in range(n_epochs_diff):
    total = 0.0; n = 0
    for x, _ in train_loader_diff:
        x = (x.to(device) * 2.0 - 1.0)        # [0,1] -> [-1,1]
        t = torch.randint(0, T_mnist, (x.size(0),), device=device)
        eps = torch.randn_like(x)
        xt = q_sample_mnist(x, t, eps)
        eps_pred = diffuser(xt, t)
        loss = F.mse_loss(eps_pred, eps)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * x.size(0); n += x.size(0)
    print(f"Epoch {epoch+1}: mean noise-prediction MSE = {total/n:.4f}")

In [ ]:
# Sample 16 digits
@torch.no_grad()
def sample_mnist(model, n=16):
    x = torch.randn(n, 784, device=device)
    for t in reversed(range(T_mnist)):
        t_b = torch.full((n,), t, dtype=torch.long, device=device)
        eps_pred = model(x, t_b)
        coef1 = 1.0 / torch.sqrt(alphas_m[t])
        coef2 = betas_m[t] / torch.sqrt(1.0 - alpha_bars_m[t])
        x_mean = coef1 * (x - coef2 * eps_pred)
        if t > 0:
            x = x_mean + torch.sqrt(betas_m[t]) * torch.randn_like(x)
        else:
            x = x_mean
    return x.cpu().numpy()

diffuser.eval()
samples = sample_mnist(diffuser, n=16)
# Re-scale back to [0, 1] for display
samples = np.clip((samples + 1.0) / 2.0, 0.0, 1.0).reshape(16, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
for ax, img in zip(axes.flat, samples):
    ax.imshow(img, cmap="gray"); ax.axis("off")
plt.suptitle("Diffusion samples after 3 epochs of training"); plt.show()

Many of the samples are recognizable as digits; a few are noise or hybrid shapes. With more training, a U-Net architecture instead of an MLP, and conditioning on the digit class, this is essentially the recipe behind every modern image generator. The mathematics --- noise schedule, MSE on noise prediction, reverse sampling --- is identical.

---

## What you have built

- The maximum-likelihood and KL-divergence machinery as concrete computations.
- A working VAE on MNIST with a 2D latent space you can plot and walk through.
- A working denoising diffusion model from scratch on a 2D toy dataset, with side-by-side visualization of forward noising and reverse generation.
- A small diffusion model on MNIST that produces recognizable digits after a few minutes of training.

Three core ideas in one notebook: likelihood, ELBO, denoising score. Every generative model used in production today is some variation on one of these three. Day 4 asks how the architecture itself --- not just the loss --- should match the structure of the data.